In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import numpy as np
import os
import polars as pl
from typing import Dict, List, Set, Tuple


Setting cofigurations and paths for this notebook. Since this notebook was run in Google Colab, the paths are set accordingly.

In [ ]:
class Config:
  TRAIN_PATH = "/content/drive/MyDrive/dataset/train.parquet"
  BENCHMARK_QUERIES_PATH = "/content/drive/MyDrive/dataset/benchmark_queries.parquet"
  BENCHMARK_ITEMS_PATH = "/content/drive/MyDrive/dataset/benchmark_items.parquet"

  # Validation split parameters
  VAL_NUM_QUERIES = 2452
  VAL_CORPUS_SIZE = 189212
  VAL_DIR = "/content/drive/MyDrive/dataset/val_split"

  RANDOM_SEED = 42

# EDA

First, we would like to learn what data we are working with.

In [ ]:
train_df = pl.read_parquet(Config.TRAIN_PATH)
bench_q_df = pl.read_parquet(Config.BENCHMARK_QUERIES_PATH)
bench_i_df = pl.read_parquet(Config.BENCHMARK_ITEMS_PATH)

In [ ]:
train_df.head()

search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
str,i64,i32,str,i64,str,f64,f64,"decimal[27,15]",i64,"decimal[18,15]",i64,"decimal[17,15]",bool,bool,str,str,str,i64
"""скупка телевизоров""",652430,0,"""""",114,"""Скупка б/у техники""",91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,true,false,"""Вид услуги Место оказания услу…","""e8b685dffe1a408e""","""Скупаю практически любую совре…",114
"""автоподбор""",640860,0,"""Рейтинг пользователя 4 звезды …",114,"""Автоподбор Разовый осмотр авто…",462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,false,false,"""Вид услуги Место оказания услу…","""92e1b0446f827b59""","""🚗 Автоподбор и выездная диагно…",114
"""баня на дровах""",653240,0,"""Онлайн-запись Тип услуги СПА-у…",114,"""Баня на дровах ""Прованс"" на Цв…",null,null,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,false,false,"""Вид услуги Красота, здоровье М…","""624846856ce81d69""","""В ритме современной жизни так…",114
"""изготовление госномера на авто""",634670,0,"""Вид услуги Оборудование, произ…",114,"""Изготовление дубликатов авто н…",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,false,false,"""Вид услуги Оборудование, произ…","""45b8628b9c6e7b85""","""Изготовим дубликат номера по у…",114
"""укладка плитки""",658430,0,"""Тип услуги Ремонт квартир и до…",114,"""Ремонт и отделка квартир под к…",1.0,5.0,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,false,false,"""Вид услуги Ремонт и отделка Ти…","""c95a4a7daf2a967f""","""отделочные работы любой сложно…",114


Check nulls

In [ ]:
train_df.null_count()

search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,19211,28232,0,0,4,0,4,0,0,0,0,0,0


In [ ]:
null_counts = train_df.null_count()
for col in train_df.columns:
    count = null_counts[col][0]
    percentage = count / train_df.height * 100
    if count > 0:
      print(f'Column {col}: {count} nulls, {percentage:.4f} % of total')

Column item_rating_reviews_count: 19211 nulls, 3.8602 % of total
Column item_rating: 28232 nulls, 5.6728 % of total
Column item_longitude: 4 nulls, 0.0008 % of total
Column item_latitude: 4 nulls, 0.0008 % of total


In [ ]:
train_unique_q = train_df.select(pl.col("search_query").n_unique()).item()
bench_unique_q = bench_q_df.select(pl.col("search_query").n_unique()).item()

In [ ]:
print(f"Number of unique queries in train: {train_unique_q} out of {train_df.height}")
print(f"Number of unique queries in benchmark: {bench_unique_q} out of {bench_q_df.height}")

Number of unique queries in train: 74529 out of 497673
Number of unique queries in benchmark: 2452 out of 2452


In [ ]:
train_q_set = set(train_df.select("search_query")["search_query"].to_list())
bench_q_set = set(bench_q_df.select("search_query")["search_query"].to_list())

In [ ]:
q_overlap = bench_q_set.intersection(train_q_set)

In [ ]:
print(f'{len(q_overlap)} queries out of {len(bench_q_set)} benchmark queries or {(len(q_overlap) / len(bench_q_set)):.4%} are in train_df')

907 queries out of 2452 benchmark queries or 36.9902% are in train_df


In [ ]:
delivery_share = train_df.filter(pl.col("search_is_delivery_search") == 1).height / train_df.height
delivery_share

2.4112218263799723e-05

In [ ]:
same_location_share = train_df.filter(pl.col("search_location_id") == pl.col("item_location_id")).height / train_df.height
same_location_share

0.8310195650557696

In [ ]:
print(f'Delivery is on: {delivery_share:.4%}')
print(f'Identical location: {same_location_share:.4%}')

Delivery is on: 0.0024%
Identical location: 83.1020%
